In [1]:
pip install torch torchvision timm tqdm

  Using cached timm-1.0.24-py3-none-any.whl.metadata (38 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached safetensors-0.7.0-cp38-abi3-macosx_11_0_arm64.whl.metadata (4.1 kB)
  Using cached hf_xet-1.2.0-cp37-abi3-macosx_11_0_arm64.whl.metadata (4.9 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached markupsafe-3.0.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (2.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 MB 3.0 MB/s  0:00:26m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 2.2 MB/s  0:00:00 eta 0:00:01
Using cached timm-1.0.24-py3-none-any.whl (2.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 2.2 MB/s  0:00:00 eta 0:00:01
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 4.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.3/553.3 kB 3.8 MB/s  0

In [3]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm
from tqdm import tqdm
import os

/Users/erroldmello/Documents/college/pawlensai/roboflowvenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# ---------------- CONFIG ----------------
DATA_DIR = "dataset"
BATCH_SIZE = 16
EPOCHS = 5
LR = 3e-4
NUM_CLASSES = 6
IMAGE_SIZE = 224
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CLASS_NAMES = [
    "demodicosis",
    "dermatitis",
    "fungal_infections",
    "healthy",
    "hypersensitivity",
    "ringworm"
]


In [5]:
# ---------------- TRANSFORMS ----------------
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])


In [7]:
# ---------------- DATASETS ----------------
train_ds = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), transform=train_transform)
val_ds   = datasets.ImageFolder(os.path.join(DATA_DIR, "valid"), transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)


In [8]:
# ---------------- MODEL ----------------
model = timm.create_model(
    "vit_base_patch16_224",
    pretrained=True,
    num_classes=NUM_CLASSES
)

model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

In [6]:
# ---------------- TRAIN LOOP ----------------
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Validation
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total
    print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f}")


Epoch 1/5: 100%|██████████| 185/185 [07:49<00:00,  2.54s/it]


Epoch 1 | Train Loss: 331.6451 | Val Acc: 0.3610


Epoch 2/5: 100%|██████████| 185/185 [07:23<00:00,  2.39s/it]


Epoch 2 | Train Loss: 262.8490 | Val Acc: 0.3832


Epoch 3/5: 100%|██████████| 185/185 [07:30<00:00,  2.44s/it]


Epoch 3 | Train Loss: 217.2748 | Val Acc: 0.6075


Epoch 4/5: 100%|██████████| 185/185 [07:33<00:00,  2.45s/it]


Epoch 4 | Train Loss: 193.7121 | Val Acc: 0.6706


Epoch 5/5: 100%|██████████| 185/185 [07:32<00:00,  2.44s/it]


Epoch 5 | Train Loss: 183.9082 | Val Acc: 0.6928


In [7]:
# ---------------- SAVE MODEL ----------------
torch.save(model.state_dict(), "vit_skin_disease.pth")

In [9]:
# ---------------- TEST DATASET ----------------
test_ds = datasets.ImageFolder(
    os.path.join(DATA_DIR, "test"),
    transform=val_transform   # use validation transforms (NO augmentation)
)

test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)


In [10]:
# ---------------- TEST EVALUATION ----------------
# Load saved weights if running this cell independently
checkpoint_path = "vit_skin_disease.pth"
if os.path.exists(checkpoint_path):
    state = torch.load(checkpoint_path, map_location=DEVICE)
    try:
        model.load_state_dict(state)
    except Exception:
        # if saved was a full model, try loading directly
        model = state
    model.to(DEVICE)
    print(f"Loaded checkpoint: {checkpoint_path}")
else:
    print(f"Warning: checkpoint not found at {checkpoint_path}. Using current model weights.")

model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        preds = outputs.argmax(dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

test_acc = correct / total if total > 0 else 0.0
print(f"\nTest Accuracy: {test_acc:.4f}")


Loaded checkpoint: vit_skin_disease.pth

Test Accuracy: 0.6752


In [12]:
# ---------------- PER-CLASS ACCURACY & CONFUSION MATRIX ----------------
import numpy as np
import torch

# collect predictions and labels over the test set
all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu()

        all_preds.append(preds)
        all_labels.append(labels.cpu())

if len(all_preds) == 0:
    print("No predictions collected from test_loader.")
else:
    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    num_classes = len(CLASS_NAMES) if 'CLASS_NAMES' in globals() \
        else int(max(all_labels.max(), all_preds.max()) + 1)

    # Build confusion matrix
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(all_labels, all_preds):
        cm[int(t), int(p)] += 1

    # Per-class accuracy
    per_class_acc = []
    for i in range(num_classes):
        total = cm[i].sum()
        correct = cm[i, i]
        acc = float(correct) / total if total > 0 else 0.0
        per_class_acc.append(acc)

    print("\nPer-class accuracy:")
    for i, acc in enumerate(per_class_acc):
        name = CLASS_NAMES[i] if 'CLASS_NAMES' in globals() and i < len(CLASS_NAMES) else str(i)
        print(f"  {name}: {acc:.4f} ({cm[i,i]}/{cm[i].sum()})")

    print("\nConfusion matrix (rows=true, cols=pred):")
    print(cm)

    # Optional: classification report if sklearn available
    try:
        from sklearn.metrics import classification_report

        target_names = CLASS_NAMES if 'CLASS_NAMES' in globals() else None
        print("\nClassification report:")
        print(classification_report(
            all_labels,
            all_preds,
            target_names=target_names,
            zero_division=0
        ))
    except Exception as e:
        print("\nCould not generate classification report:", e)



Per-class accuracy:
  demodicosis: 0.7121 (47/66)
  dermatitis: 0.5283 (28/53)
  fungal_infections: 0.5652 (39/69)
  healthy: 0.0000 (0/28)
  hypersensitivity: 0.8100 (81/100)
  ringworm: 0.8348 (96/115)

Confusion matrix (rows=true, cols=pred):
[[47  9  2  0  5  3]
 [ 9 28  3  0  5  8]
 [ 2 24 39  0  4  0]
 [ 4  6  1  0 13  4]
 [ 4  2  4  0 81  9]
 [13  3  0  0  3 96]]

Classification report:
                   precision    recall  f1-score   support

      demodicosis       0.59      0.71      0.65        66
       dermatitis       0.39      0.53      0.45        53
fungal_infections       0.80      0.57      0.66        69
          healthy       0.00      0.00      0.00        28
 hypersensitivity       0.73      0.81      0.77       100
         ringworm       0.80      0.83      0.82       115

         accuracy                           0.68       431
        macro avg       0.55      0.58      0.56       431
     weighted avg       0.65      0.68      0.66       431

